I installed and imported the Pandas library.

In [1]:
!pip install pandas

zsh:1: /Users/cansu/Projects/Data-Business-Analytics-Portfolio/.venv/bin/pip: bad interpreter: /Users/cansu/Desktop/Data-Business-Analytics-Portfolio/.venv/bin/python: no such file or directory


In [2]:
import pandas as pd

I have uploaded all the Olist and Marketing Funnel tables to be used in the Nova7 scenario.

In [3]:
customers = pd.read_csv("../data/raw_data/olist/olist_customers_dataset.csv")
geo_locations = pd.read_csv("../data/raw_data/olist/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw_data/olist/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw_data/olist/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw_data/olist/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw_data/olist/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw_data/olist/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw_data/olist/olist_sellers_dataset.csv")
product_category_name_translations = pd.read_csv("../data/raw_data/olist/product_category_name_translation.csv")
closed_deals = pd.read_csv("../data/raw_data/marketing/olist_closed_deals_dataset.csv")
marketing_leads = pd.read_csv("../data/raw_data/marketing/olist_marketing_qualified_leads_dataset.csv")

In [4]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [5]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


The number of customer_unique_ids is less than the total number of customer_ids. This means there are multiple rows with some identical unique_ids.

The `customer_zip_code_prefix` field is stored as an int64, but this is a categorical/descriptive field, not a numerical measure. For example, aggregate statistics like mean/standard deviation are meaningless here. Therefore, I will convert it to a string during the cleanup phase.

In [6]:
customers.describe(include="all")

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
count,99441,99441,99441.000000,99441,99441
unique,99441,96096,NaN,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,NaN,sao paulo,SP
freq,1,17,NaN,15540,41746
mean,NaN,NaN,35137.474583,NaN,NaN
std,NaN,NaN,29797.938996,NaN,NaN
min,NaN,NaN,1003.000000,NaN,NaN
25%,NaN,NaN,11347.000000,NaN,NaN
50%,NaN,NaN,24416.000000,NaN,NaN
75%,NaN,NaN,58900.000000,NaN,NaN


This code shows how many different rows (i.e., how many different customer_IDs) each customer_unique_id appears in the customers table. For example, the actual customer named 8d50f5eadf50201ccdcedfb9e2ac8455 is registered with 17 different customer_IDs in the table. A single customer appears to have multiple customer records. The underlying business reason is currently unknown and requires further investigation. For example, the same person may have placed orders using different email addresses, or a customer may have deleted their account and then registered again; such business rules may apply.

In [7]:
customers.customer_unique_id.value_counts().sort_values(ascending=False)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
ca77025e7201e3b30c44b472ff346268     7
6469f99c1f9dfae7733b25662e7f1782     7
                                    ..
26c602fff4586cb473c2c37abe87caef     1
99fefcc024154d6088c096ff42edb1cd     1
edf60415af3f7ec6da031ebbb8abb471     1
56c964ce504f3dce08f3f1df858eccd6     1
84732c5050c01db9b23e19ba39899398     1
Name: count, Length: 96096, dtype: int64

I answered the question "How many customer_unique_id instances occur more than once?" in the relevant sections.

In [8]:
repeat_counts = customers['customer_unique_id'].value_counts()
(repeat_counts > 1).sum()

np.int64(2997)

Approximately 3% of Nova7's total unique customers appear under multiple customer_IDs in the system. This indicates that we should use customer_unique_ID instead of customer_ID in "repeat customer" analytics; otherwise, we might mistakenly count the same person as multiple different customers.

In [9]:
repeat_rate = (repeat_counts > 1).sum() / customers['customer_unique_id'].nunique() * 100
repeat_rate

np.float64(3.1187562437562435)

In [10]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [11]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [12]:
orders.describe()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-03-31 15:08:21,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


Most of the orders have been delivered. However, there are 7 more cases besides those. I checked the numbers for each one.

In [13]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Some orders with blank delivery dates have been shipped, some have been cancelled, some are out of stock, etc. However, 8 orders show "delivered" but the "order_delivered_customer_date" section is empty. This is a data quality issue. There may be missing or incorrectly entered data.

In [14]:
orders[orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

I'm curious about the date range covered by the dataset. I might need this for time-based trend analysis. Here, I converted the string to datetime because it's more accurate to work with date columns as datetimes.

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_purchase_timestamp'].min(), orders['order_purchase_timestamp'].max()

I examined the number of orders per month. This part is important for analyzing how order volume progresses. Some months show a significant decrease compared to others. There may have been an interruption during the data collection period. If so, I will remove these periods from the trend analysis. It's important that we notice these things, otherwise misinterpretations can occur. 

In [15]:
orders['order_purchase_timestamp'].dt.to_period('M').value_counts().sort_index()

AttributeError: Can only use .dt accessor with datetimelike values

In [ ]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [ ]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


The table has 112,650 rows, more than the number of rows in the orders table. This is because an order can contain multiple products, and therefore order_items has more rows.

The order_item_id value is max=21. This means there are 21 different products in one order. This could be a bulk purchase.

In the price column, the standard deviation is even larger than the mean itself. This means the data isn't evenly distributed around the mean; it's spread over a wide range. A right-skewed distribution is present. Most products are cheap, but a small number of expensive products are pushing the mean and standard deviation upwards.

In such cases, the median is more reliable than the mean. When I create a visualization of the average value of orders in the future, I should not ignore these values.

In [ ]:
order_items.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


There are orders where the freight_value min = 0.00, meaning the shipping cost is zero. This might be a "free shipping" order, but I'm checking anyway.
There are 383 orders like this, and based on my dataset, it's a logical scenario.

In [ ]:
(order_items['freight_value'] == 0).sum()

np.int64(383)

In [ ]:
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [ ]:
order_payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


The max=29 for `payment_sequential` is a noteworthy number. Different payment methods might have been used, but 29 doesn't seem very realistic. The system might have given an error, the customer might have tried repeatedly, the system might have split the payment, or it could be a data structure issue.

Again, there's a large difference between the maximum payout amount and the average, and the standard deviation remains above the average. As above, payouts are generally in the lower range, but some are higher than the average. I can confirm this from the 75% figure as well.

Order IDs can be repeated in this table. In fact, this table has more rows than the orders table because an order can be paid for in multiple ways, and a separate record is created for each payment method.

In [ ]:
order_payments.describe()

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


There can be multiple reasons for a high `payment_sequential` value. To understand this, I examined the payment type in cases where there were more than 10 sequential payments. According to this output, a customer used multiple coupons in the same order, and the system recorded each coupon as a separate payment.

In [ ]:
order_payments[order_payments['payment_sequential'] > 10]['payment_type'].value_counts()

payment_type
voucher    127
Name: count, dtype: int64

I wanted to examine the order with 29 payment records separately.

In [ ]:
order_payments.groupby("order_id").size().sort_values(ascending=False)

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
fedcd9f7ccdc8cba3a18defedd1a5547    19
                                    ..
56bd45163229b35ca0ab490c1e3d3233     1
56bc98e6d5b88c2cdb905f2fbec2ca3a     1
56bbc7d92e6e74b8782abbf5ee336a92     1
56bafc014f8ed2f34cfe598592c65fd8     1
fffe41c64501cc87c801fd61db3f6244     1
Length: 99440, dtype: int64

As you can see, all records are of the voucher type. There are two 0.00 values, and the amounts are different from each other. These 0.00 values ​​could have several causes. It could be a cancelled voucher, a technical issue, or a rounding error.

In [ ]:
order_payments[
    order_payments["order_id"] == "fa65dad1b0e818e3ccc5cb0e39231352"
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
4885,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
24879,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.68
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
32519,fa65dad1b0e818e3ccc5cb0e39231352,11,voucher,1,4.03


In [ ]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [ ]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


Mostly rated 5 stars. However, this dataset only contains customers who submitted reviews, so it may not represent the satisfaction of all customers.

In [ ]:
order_reviews.describe()

,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


Review scores show a polarized (bimodal-leaning) pattern: 5-star is most common, but 1-star ranks third — more common than 2-3 star. This is a typical pattern in customer feedback, where neutral experiences are less likely to prompt a review than very positive or very negative ones.

In [ ]:
order_reviews.review_score.value_counts()

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

The `order_reviews` table normally has 99,225 rows, but there are 98,673 different `order_id` in the table. This suggests that some orders have multiple reviews. This must be handled carefully during merging — a naive merge could duplicate order-level data. Will likely take the latest review per order, or aggregate review scores, during the cleaning phase.

In [ ]:
order_reviews['order_id'].nunique()

98673

In [ ]:
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

Logically, the `review_answer_timestamp` should always come after the `review_creation_date`. This is because a reply to a comment comes after the comment is written. I tested this here, and it's consistent. If it weren't 0, there might be a data quality anomaly.

In [ ]:
(order_reviews['review_answer_timestamp'] < order_reviews['review_creation_date']).sum()

np.int64(0)

I examined how many days it typically takes to respond to a review. The average is 2.5 days, but the standard deviation is very high (9.89) — this is driven by extreme outliers, with a maximum of 518 days. Since review_answer_timestamp has no nulls, every review does have a recorded response — but some responses took an unusually long time, possibly due to delayed processing or batch handling on the platform's side. Looking at the quartiles (25%: 1 day, 50%: 1 day, 75%: 3 days), the typical response time is much shorter than the mean suggests — again, median is more representative here than mean.

In [ ]:
order_reviews['response_time_days'] = (order_reviews['review_answer_timestamp'] - order_reviews['review_creation_date']).dt.days
order_reviews['response_time_days'].describe()

count    99224.000000
mean         2.582248
std          9.890526
min          0.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        518.000000
Name: response_time_days, dtype: float64

In [ ]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In the original dataset, the column name was misspelled as "lenght", it should have been "length".

In [ ]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [ ]:
products.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


The minimum value entered for weight appears to be "0". I checked how many of these there are.

In [ ]:
(products['product_weight_g'] == 0).sum()

np.int64(4)

I checked what these four products are. They all belong to the same category: "bed, table, bathroom". The weight field might have been overlooked during data entry, a standard template might have been used, or the system might have accepted 0 if it wasn't a required entry. I will correct this during data cleaning.

In [ ]:
products[products['product_weight_g'] == 0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


Some rows contain 610 null values, while others contain only 2. Using this code, I calculated the number of rows that are empty simultaneously in both the "product_category_name" and "product_weight_g" columns. Since there's only one, these two issues are likely unrelated. This simplifies the cleanup process.

In [ ]:
products[products['product_category_name'].isnull()].shape[0]
products[products['product_weight_g'].isnull()].shape[0]
products[products['product_category_name'].isnull() & products['product_weight_g'].isnull()].shape[0]

1

In [ ]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [ ]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


In [ ]:
sellers.describe()

,seller_zip_code_prefix
count,3095.000000
mean,32291.059451
std,32713.453830
min,1001.000000
25%,7093.500000
50%,14940.000000
75%,64552.500000
max,99730.000000


This shows which states the sellers are concentrated in. The state of SP is dominant on both the buyer and seller sides.

In [ ]:
sellers['seller_state'].value_counts()

seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64

sdr_id and sr_id; the IDs of the sales representatives who closed the deal, namely the Sales Development Rep and the Sales Rep.

won_date; the date the deal was closed/won.

business_segment, business_type; the seller's business type and sector.

has_company, has_gtin; fields such as "does the company have registration, does it have a product barcode system (GTIN)?".

declared_product_catalog_size, declared_monthly_revenue — the seller's self-declared catalog size and monthly revenue.

In [ ]:
closed_deals.head()

,mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue
0,5420aad7fec3549a85876ba1c529bd84,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0
1,a555fb36b9368110ede0f043dfc3b9a0,bbb7d7893a450660432ea6652310ebb7,09285259593c61296eef10c734121d5b,d3d1e91a157ea7f90548eef82f1955e3,2018-05-08 20:17:59,car_accessories,industry,eagle,NaN,NaN,NaN,reseller,NaN,0.0
2,327174d3648a2d047e8940d7d15204ca,612170e34b97004b3ba37eae81836b4c,b90f87164b5f8c2cfa5c8572834dbe3f,6565aa9ce3178a5caf6171827af3a9ba,2018-06-05 17:27:23,home_appliances,online_big,cat,NaN,NaN,NaN,reseller,NaN,0.0
3,f5fee8f7da74f4887f5bcae2bafb6dd6,21e1781e36faf92725dde4730a88ca0f,56bf83c4bb35763a51c2baab501b4c67,d3d1e91a157ea7f90548eef82f1955e3,2018-01-17 13:51:03,food_drink,online_small,NaN,NaN,NaN,NaN,reseller,NaN,0.0
4,ffe640179b554e295c167a2f6be528e0,ed8cb7b190ceb6067227478e48cf8dde,4b339f9567d060bcea4f5136b9f5949e,d3d1e91a157ea7f90548eef82f1955e3,2018-07-03 20:17:45,home_appliances,industry,wolf,NaN,NaN,NaN,manufacturer,NaN,0.0


The columns `has_company`, `has_gtin`, and `declared_product_catalog_size` contain many null values. This is a high number, so it's probably not a random omission. They could be optional form questions or fields added later and recently asked. This missing data issue needs to be addressed in the analytics section.

In [ ]:
closed_deals.info()

<class 'pandas.DataFrame'>
RangeIndex: 842 entries, 0 to 841
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   mql_id                         842 non-null    str    
 1   seller_id                      842 non-null    str    
 2   sdr_id                         842 non-null    str    
 3   sr_id                          842 non-null    str    
 4   won_date                       842 non-null    str    
 5   business_segment               841 non-null    str    
 6   lead_type                      836 non-null    str    
 7   lead_behaviour_profile         665 non-null    str    
 8   has_company                    63 non-null     object 
 9   has_gtin                       64 non-null     object 
 10  average_stock                  66 non-null     str    
 11  business_type                  832 non-null    str    
 12  declared_product_catalog_size  69 non-null     float64
 13  d

declared_monthly_revenue is heavily skewed: median is 0, mean is ~73K, but max is 50M — a small number of large sellers pull the average far above the typical value. Zero may mean 'no revenue yet' (new sellers) or could be a placeholder for 'not declared' — this ambiguity will be noted, not resolved, since we can't confirm from the data alone.

In [ ]:
closed_deals.describe()

,declared_product_catalog_size,declared_monthly_revenue
count,69.000000,8.420000e+02
mean,233.028986,7.337768e+04
std,352.380558,1.744799e+06
min,1.000000,0.000000e+00
25%,30.000000,0.000000e+00
50%,100.000000,0.000000e+00
75%,300.000000,0.000000e+00
max,2000.000000,5.000000e+07


In [ ]:
marketing_leads.head()

,mql_id,first_contact_date,landing_page_id,origin
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search


`first_contact_date` is the date the lead first made contact, `landing_page_id` is the landing page they came from, and `origin` is the marketing source.

In [ ]:
marketing_leads.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   mql_id              8000 non-null   str  
 1   first_contact_date  8000 non-null   str  
 2   landing_page_id     8000 non-null   str  
 3   origin              7940 non-null   str  
dtypes: str(4)
memory usage: 250.1 KB


In [ ]:
marketing_leads.describe()

,mql_id,first_contact_date,landing_page_id,origin
count,8000,8000,8000,7940
unique,8000,336,495,10
top,dac32acd4db4c29c230538b72f8dd87d,2018-05-02,b76ef37428e6799c421989521c0e5077,organic_search
freq,1,93,912,2296


This checks if all mql_id's in closed_deals actually exist in the marketing_leads table. If it returns True, the relationship between the two tables is consistent, so I can confidently join them.

If it returns False, some closed_deals would be missing source lead records.

In [ ]:
closed_deals['mql_id'].isin(marketing_leads['mql_id']).all()

np.True_

The marketing_leads table has 8000 rows, while the closed_deals table has 842 rows. This means some of them have converted to resellers. This allows me to calculate the conversion rate.

Approximately 1 out of every 10 leads captured by marketing turns into an actual salesperson.

In [ ]:
conversion_rate = closed_deals['mql_id'].nunique() / marketing_leads['mql_id'].nunique() * 100
conversion_rate

10.525

customer_id is unique across all 99,441 records and can safely be treated as the primary key of the customers table.

In [ ]:
customers["customer_id"].nunique(), len(customers)

(99441, 99441)

"Do all the customer_ids in the Orders table actually exist in the Customers table?" I'm doing the test.

In [ ]:
orders["customer_id"].isin(customers["customer_id"]).all()

np.True_

All six records have an order_status of canceled despite having a delivery date. This may indicate post-delivery cancellations, returns, or potential data quality issues. Additional business context would be required before treating these records as errors.

In [ ]:
non_delivered = orders[
    orders["order_status"] != "delivered"
]
non_delivered

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
44,ee64d42b8cf066f35eac1cf57de1aa85,caded193e8e47b8362864762a83db3c5,shipped,2018-06-04 16:44:48,2018-06-05 04:31:18,2018-06-05 14:32:00,NaN,2018-06-28 00:00:00
103,0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03 17:44:42,2018-08-07 06:15:14,NaN,NaN,2018-08-21 00:00:00
128,15bed8e2fec7fdbadb186b57c46c92f2,f3f0e613e0bdb9c7cee75504f0f90679,processing,2017-09-03 14:22:03,2017-09-03 14:30:09,NaN,NaN,2017-10-03 00:00:00
154,6942b8da583c2f9957e990d028607019,52006a9383bf149a4fb24226b173106f,shipped,2018-01-10 11:33:07,2018-01-11 02:32:30,2018-01-11 19:39:23,NaN,2018-02-07 00:00:00
...,...,...,...,...,...,...,...,...
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaN,NaN,NaN,2018-10-01 00:00:00
99313,e9e64a17afa9653aacf2616d94c005b8,b4cd0522e632e481f8eaf766a2646e86,processing,2018-01-05 23:07:24,2018-01-09 07:18:05,NaN,NaN,2018-02-06 00:00:00
99347,a89abace0dcc01eeb267a9660b5ac126,2f0524a7b1b3845a1a57fcf3910c4333,canceled,2018-09-06 18:45:47,NaN,NaN,NaN,2018-09-27 00:00:00
99348,a69ba794cc7deb415c3e15a0a3877e69,726f0894b5becdf952ea537d5266e543,unavailable,2017-08-23 16:28:04,2017-08-28 15:44:47,NaN,NaN,2017-09-15 00:00:00


I'm checking to see if there are any orders with a confirmation date prior to the purchase data. That would be against business rules.

In [ ]:
orders[orders["order_approved_at"] < orders["order_purchase_timestamp"]]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


I'm checking to see if there are any orders where reviews were submitted before delivery. There could be a few reasons for this. The store might allow reviews before delivery, or the review_creation_date might not be the exact time the customer wrote the review; it could be a system record date.

In [ ]:
orders_reviews = pd.merge(
    orders,
    order_reviews,
    on="order_id",
    how="inner"
)
review_before_delivery = orders_reviews[
    orders_reviews["review_creation_date"]
    < orders_reviews["order_delivered_customer_date"]
]

review_before_delivery
orders_reviews["review_creation_date"] = pd.to_datetime(
    orders_reviews["review_creation_date"]
)

orders_reviews["order_delivered_customer_date"] = pd.to_datetime(
    orders_reviews["order_delivered_customer_date"]
)
review_before_delivery

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,response_time_days
19,203096f03d82e0dffbc41ebc2e2bcfb7,d2b091571da224a1b36412c18bc3bbfe,delivered,2017-09-18 14:31:30,2017-09-19 04:04:09,2017-10-06 17:50:03,2017-10-09 22:23:46,2017-09-28 00:00:00,38cae21b1b57a95959440380d5b2ef7a,2,NaN,os correios estäo em greve... näo recebi nenhu...,2017-10-01,2017-10-01 17:55:21,0
24,fbf9ac61453ac646ce8ad9783d7d0af6,3a874b4d4c4b6543206ff5d89287f0c3,delivered,2018-02-20 23:46:53,2018-02-22 02:30:46,2018-02-26 22:25:22,2018-03-21 22:03:54,2018-03-12 00:00:00,6a1a8e54de03ab98e6e8ff56e56e507f,2,NaN,Demora muito entregar. Já passou o prazo e ain...,2018-03-16,2018-03-20 23:10:58,4
34,8563039e855156e48fccee4d611a3196,5f16605299d698660e0606f7eae2d2f9,delivered,2018-02-17 15:59:46,2018-02-17 16:15:34,2018-02-20 23:03:56,2018-03-20 00:59:25,2018-03-20 00:00:00,f121467a10eee0929f364c7d62abc9b5,5,NaN,há muito tempo efetuo compras atraves desta lo...,2018-03-20,2018-03-23 22:56:05,3
40,6ea2f835b4556291ffdc53fa0b3b95e8,c7340080e394356141681bd4c9b8fe31,delivered,2017-11-24 21:27:48,2017-11-25 00:21:09,2017-12-13 21:14:05,2017-12-28 18:59:23,2017-12-21 00:00:00,5caca29ffffe9086162ca51303817420,1,NaN,"Inicialmente, na data da compra o produto era ...",2017-12-22,2017-12-28 11:25:32,6
57,a685d016c8a26f71a0bb67821070e398,911e4c37f5cafe1604fe6767034bf1ae,delivered,2017-03-13 18:14:36,2017-03-13 18:14:36,2017-03-22 14:03:09,2017-04-06 13:37:16,2017-03-30 00:00:00,19d60869413aa63834affc9ecc9d9f90,1,NaN,Não recebi,2017-04-02,2017-04-02 10:58:38,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99180,0fa1fab1d7c1211c824596ed5e111e3c,7f3bd6c94d2daf7b6462d1a894a775b4,delivered,2018-03-13 21:48:57,2018-03-13 22:40:28,2018-03-14 19:27:23,2018-04-05 19:59:49,2018-04-02 00:00:00,d6e89c99dd004e190fb802797253e9ba,1,NaN,"Nao volto a comprar com esta loja , demorando ...",2018-04-04,2018-04-04 10:15:42,0
99187,a2a701c6f01ddffde8a1bde136ed7d4a,8543703cb2bc95c3606af4af727d604f,delivered,2017-11-26 10:26:55,2017-11-26 10:36:06,2017-11-27 22:49:48,2017-12-16 02:54:56,2018-01-03 00:00:00,6bb2aaa269806b7a0f83f4ee1a346887,4,NaN,NaN,2017-12-16,2017-12-18 21:24:40,2
99206,38e9133ce29f6bbe35aed9c3863dce01,ad312389a098ceff46ce92c4595c06d0,delivered,2017-10-12 20:54:11,2017-10-14 03:28:24,2017-10-17 17:04:42,2017-11-21 17:06:59,2017-10-31 00:00:00,4a57b77d844594f24904a568218a2d96,1,NaN,nao recebi o produto que ja paguei ! nao recom...,2017-11-03,2017-11-06 18:50:22,3
99208,d692ef54145c9cb3322ec2e5508aa3f4,82ddfcf9438b0cd1117b55ac33184df8,delivered,2018-03-21 19:47:18,2018-03-21 20:05:26,2018-03-22 21:11:58,2018-04-11 00:48:31,2018-04-09 00:00:00,6b2ee488cd87d8a9ee67eb63ad5a7a4c,1,NaN,"Prateleiras com cantos vivo, colunas nao encai...",2018-04-11,2018-04-11 12:41:03,0


I review orders shipped before approval. Reasons for delays could include timestamp issues, the automated approval process, the ETL process for the dataset, or a genuine data problem.

In [ ]:
carrier_check = orders[
    orders["order_approved_at"].notna() &
    orders["order_delivered_carrier_date"].notna()
]
carrier_before_approval = carrier_check[
    carrier_check["order_delivered_carrier_date"] <
    carrier_check["order_approved_at"]
]

carrier_before_approval

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04 00:00:00
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15 00:00:00
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31 00:00:00
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31 00:00:00
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06 00:00:00
...,...,...,...,...,...,...,...,...
99091,240ead1a7284667e0ec71d01f80e4d5e,fcdd7556401aaa1c980f8b67a69f95dc,delivered,2018-07-02 16:30:02,2018-07-05 16:17:59,2018-07-05 14:11:00,2018-07-10 23:21:47,2018-07-24 00:00:00
99230,78008d03bd8ef7fcf1568728b316553c,043e3254e68daf7256bda1c9c03c2286,delivered,2018-07-03 13:11:13,2018-07-05 16:32:52,2018-07-03 12:57:00,2018-07-10 17:47:39,2018-07-23 00:00:00
99266,76a948cd55bf22799753720d4545dd2d,3f20a07b28aa252d0502fe7f7eb030a9,delivered,2018-01-30 02:41:30,2018-02-04 23:31:46,2018-01-31 18:11:58,2018-03-18 20:08:50,2018-03-02 00:00:00
99377,a6bd1f93b7ff72cc348ca07f38ec4bee,6d63fa86bd2f62908ad328325799152f,delivered,2018-04-20 17:28:40,2018-04-24 19:26:10,2018-04-23 17:18:40,2018-04-28 17:38:42,2018-05-15 00:00:00


The majority of violating records show a difference of approximately 1–2 days between the two timestamps. However, a small number of records exhibit extremely large negative delays (up to 172 days), which appear to be outliers. These observations may indicate timestamp synchronization issues, ETL inconsistencies, or exceptional business scenarios. Further investigation is required before classifying them as data quality errors.

In [ ]:
carrier_check["approval_to_carrier_delay"] = (
    carrier_check["order_delivered_carrier_date"]
    - carrier_check["order_approved_at"]
)

carrier_before_approval = carrier_check[
    carrier_check["approval_to_carrier_delay"] < pd.Timedelta(0)
]

carrier_before_approval["approval_to_carrier_delay"].describe()

TypeError: unsupported operand type(s) for -: 'str' and 'str'

The repeated occurrence of approximately 10-day negative delays suggests that these records may represent a systematic issue rather than isolated data entry mistakes. The extreme outlier (172 days) appears highly inconsistent with the expected business workflow and should be investigated separately. At this stage, these records are flagged for further review rather than being classified as data quality errors.

In [ ]:
carrier_before_approval.sort_values(
    by="approval_to_carrier_delay"
).head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_to_carrier_delay
25883,7c48bb55e8e4f7e56d412e9653db37bc,34ef6181341eb36c47fd601c46878f00,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,2018-08-07 00:00:00,-172 days +18:44:38
14562,1fab4ac9d85079b3da72a11475ae1685,f831c1fa80308975ec2b58e4877328e0,delivered,2017-09-01 19:04:22,2017-09-13 22:06:11,2017-09-04 13:10:23,2017-09-08 20:13:03,2017-09-20 00:00:00,-10 days +15:04:12
46163,0184d4ddb259e1a4cfc2871888cf97b8,09425ea1839abf2f0d289a0ff453fa21,delivered,2017-09-01 20:04:28,2017-09-13 22:17:15,2017-09-04 14:05:50,2017-09-09 15:12:44,2017-09-20 00:00:00,-10 days +15:48:35
98710,1378f9601350615613cc8832d6789c5d,988126b4ddf725d9724e4318872ea2ae,delivered,2017-09-01 20:28:02,2017-09-13 22:03:51,2017-09-04 18:07:55,2017-09-13 22:24:46,2017-09-29 00:00:00,-10 days +20:04:04
41592,8554cb37f7158cb0b082a841d24a4589,788e845925ff64c9df5d8ba40e28cf50,delivered,2017-09-01 18:40:44,2017-09-13 21:58:04,2017-09-04 19:12:19,2017-09-08 20:07:45,2017-10-02 00:00:00,-10 days +21:14:15


I've reviewed the total orders and total payments. However, the number of rows in `payment_totals` is higher than in `order_totals`. There could be several reasons for this. There might be orders that were canceled before products were added, payments might have been received but orders couldn't be created, some tables might be handled under different scopes, or certain order statuses might be causing this discrepancy.

In [ ]:
order_totals = (
    order_items
    .groupby("order_id")[["price", "freight_value"]]
    .sum()
)
order_totals

,price,freight_value
order_id,,
00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,239.90,19.93
000229ec398224ef6ca0657da4fc703e,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14
...,...,...
fffc94f6ce00a00581880bf54a75a037,299.99,43.41
fffcd46ef2263f404302a634eb57f7eb,350.00,36.53
fffce4705a9662cd70adb13d4a31832d,99.90,16.95


In [ ]:
payment_totals = (
    order_payments
    .groupby("order_id")[["payment_value"]]
    .sum()
)
payment_totals

,payment_value
order_id,
00010242fe8c5a6d1ba2dd792cb16214,72.19
00018f77f2f0320c557190d7a144bdd3,259.83
000229ec398224ef6ca0657da4fc703e,216.87
00024acbcdf0a6daa1e931b038114c75,25.78
00042b26cf59d7ce69dfabb4e55b4fd9,218.04
...,...
fffc94f6ce00a00581880bf54a75a037,343.40
fffcd46ef2263f404302a634eb57f7eb,386.53
fffce4705a9662cd70adb13d4a31832d,116.85


In [ ]:
payment_validation = pd.merge(
    order_totals,
    payment_totals,
    on="order_id",
    how="inner"
)
payment_validation.head()

,price,freight_value,payment_value
order_id,,,
00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


In [ ]:
payment_validation["expected_payment"] = (
    payment_validation["price"] +
    payment_validation["freight_value"]
)
payment_validation["difference"] = (
    payment_validation["payment_value"]
    - payment_validation["expected_payment"]
)
payment_validation["difference"].describe()

count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: difference, dtype: float64

Usually the difference is 0. Small numerical differences (e.g., ±1e-14) are attributable to floating-point precision and are not considered data quality issues. 

In [ ]:
payment_validation["difference"].value_counts().head(10)

difference
 0.000000e+00    79051
-1.421085e-14     4014
 7.105427e-15     3751
-2.842171e-14     3240
 1.421085e-14     2864
 2.842171e-14     1315
-7.105427e-15     1101
 5.684342e-14      885
-5.684342e-14      599
 3.552714e-15      484
Name: count, dtype: int64

I consider differences of less than one cent insignificant and I focus on the largest differences.

In [ ]:
payment_validation[
    payment_validation["difference"].abs() > 0.01
].sort_values(
    by="difference",
    ascending=False
).head(10)

,price,freight_value,payment_value,expected_payment,difference
order_id,,,,,
ce6d150fb29ada17d2082f4847107665,1299.00,104.66,1586.47,1403.66,182.81
6e5fe7366a2e1bfbf3257dba0af1267f,179.19,108.72,406.92,287.91,119.01
70b742795bc441e94a44a084b6d9ce7a,269.99,196.94,578.82,466.93,111.89
996c7e73600ad3723e8627ab7bef81e4,559.90,28.00,664.43,587.90,76.53
70b7e94ea46d3e8b5bc12a50186edaf0,167.88,45.27,274.84,213.15,61.69
bc2c82b0ef78d2252b6176d1972db7c9,165.00,77.01,303.02,242.01,61.01
af9ffff2ce6b3defd34fd4c78857a379,395.65,17.52,466.97,413.17,53.80
bfdb5bbb06458d600a33d61f5f287472,297.00,51.93,394.36,348.93,45.43
8d9c0dc8d5a2ce804f6b925d8f8e6c1d,209.80,44.65,293.89,254.45,39.44


A "difference between expected and actual payment" exists in 381 orders. This represents 0.39% of the total orders. This means that Price + freight value may not always be sufficient to explain the total payment. Factors such as financing fees, promotions, rounding, and additional services can cause this difference. Furthermore, in one or a few orders, the expected payment was higher. This could be due to a discount, an invisible coupon mechanism, etc.

In [ ]:
payment_validation.loc[
    payment_validation["difference"].abs() > 0.01,
    "difference"
].describe()

count    381.000000
mean       7.534961
std       16.564267
min      -51.620000
25%        0.010000
50%        3.090000
75%        9.570000
max      182.810000
Name: difference, dtype: float64

In [ ]:
order_reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

I have confirmed that all ratings are between 1 and 5. This suggests that the review system likely restricts users to predefined rating options, reducing the possibility of invalid score entries.

In [ ]:
invalid_review_scores = order_reviews[
    (order_reviews["review_score"] < 1) |
    (order_reviews["review_score"] > 5)
]

invalid_review_scores

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,response_time_days


Now I'm verifying the product dimensions. First, I'm testing to see if any negative values ​​have been entered.

In [ ]:
products[
    (products["product_weight_g"] < 0) |
    (products["product_length_cm"] < 0) |
    (products["product_height_cm"] < 0) |
    (products["product_width_cm"] < 0)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


This time I'm checking if there's a dimension/weight condition entered as 0.

Four of the products have a weight of 0 entered. This is a very low percentage. These may never have been sold, the information may not have been entered into the system and the system may be representing them this way, or the seller may have left this field blank.

In [ ]:
(products[[
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]] == 0).sum()

product_weight_g     4
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64

These products, which have a weight of 0, have other dimensions entered. They all belong to the same product category and have almost identical physical dimensions; this points to a local data entry problem rather than a widespread data quality issue.

In [ ]:
products[
    products["product_weight_g"] == 0
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


These products, which have a weight of 0, have other dimensions entered. They all belong to the same product category and have almost identical physical dimensions; this points to a local data entry problem rather than a widespread data quality issue. I will investigate this further later.

In [ ]:
products[
    products["product_weight_g"] == 0
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


The `product_id` in the `products` table is not duplicated. I have confirmed that it is a definitive primary key.

In [ ]:
products["product_id"].duplicated().sum()

np.int64(0)

However, there are products that have all these features the same except for the ID.

In [ ]:
products.duplicated(
    subset=[
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
).sum()

np.int64(695)

I'm examining these products, each with a repeating feature. If I had used the regular duplicated() instead of "keep=false", I wouldn't have been able to see the initial records.

In [ ]:
products[
    products.duplicated(
        subset=[
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ],
        keep=False
    )
].sort_values(
    by=[
        "product_category_name",
        "product_name_lenght"
    ]
)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
4205,0152cb427657428c06633fce6d721da4,alimentos,57.0,606.0,3.0,150.0,22.0,4.0,17.0
18510,1e114096c8024159923dcf2c857e4ca4,alimentos,57.0,606.0,3.0,150.0,22.0,4.0,17.0
13382,733162823b9a1dbc958b0988d32229da,artigos_de_festas,46.0,459.0,1.0,1350.0,23.0,17.0,18.0
19487,cc6a0d67ea3d63acca23c81500670843,artigos_de_festas,46.0,459.0,1.0,1350.0,23.0,17.0,18.0
2442,81b454070eecf89b21503cd5d313aa57,automotivo,50.0,2304.0,4.0,250.0,16.0,6.0,15.0
...,...,...,...,...,...,...,...,...,...
31785,fbb1cfc2810efabf3235eccf4530f4ae,NaN,NaN,NaN,NaN,800.0,35.0,15.0,20.0
31997,0bb7d5c406dd6affef3cd4f5d6744844,NaN,NaN,NaN,NaN,200.0,16.0,2.0,11.0
32315,bb1c86c0b1cd8da99fbd48d798abcca0,NaN,NaN,NaN,NaN,830.0,19.0,6.0,26.0
32436,338f1838eb9cd29b59c9c2f4ee158f15,NaN,NaN,NaN,NaN,250.0,16.0,4.0,11.0


I'm verifying the data by checking if there are any records with a payment amount below 0.

In [ ]:
order_payments[
    order_payments["payment_value"] < 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [ ]:
order_payments["payment_installments"].value_counts().sort_index()

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64

I am reviewing two orders with 0 installments. These orders have a second payment record. Both are credit card payments with payment_sequential = 2. Since this represents an extremely small fraction of the dataset, the records are flagged as potential anomalies rather than immediately treated as data quality issues. The root cause cannot be determined from the available data.

In [ ]:
order_payments.loc[
    order_payments["payment_installments"] == 0,
    [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ]
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


When I examined the `order_items` table, I noticed that the same `product_id` appears multiple times within a single order. This means the same product was purchased multiple times in the same order. This is normal. This information could have been stored in a separate column called "quantity," but then some other information like "vendor, payment type, and order status" wouldn't have been properly stored. Therefore, this is not a data quality issue. I have verified this.

In [ ]:
order_items[
    order_items.duplicated(
        subset=["order_id", "product_id"],
        keep=False
    )
].sort_values(["order_id", "product_id"])

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
13,0008288aa423d2a3f00fcb17cd7d8719,1,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2018-02-21 02:55:52,49.90,13.37
14,0008288aa423d2a3f00fcb17cd7d8719,2,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2018-02-21 02:55:52,49.90,13.37
32,00143d0f86d6fbd9f9b38ab440ac16f5,1,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20 16:07:52,21.33,15.10
33,00143d0f86d6fbd9f9b38ab440ac16f5,2,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20 16:07:52,21.33,15.10
34,00143d0f86d6fbd9f9b38ab440ac16f5,3,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20 16:07:52,21.33,15.10
...,...,...,...,...,...,...,...
112635,fff8287bbae429a99bb7e8c21d151c41,2,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-27 12:29:22,180.00,48.14
112640,fffb9224b6fc7c43ebb0904318b10b5f,1,43423cdffde7fda63d0414ed38c11a73,b1fc4f64df5a0e8b6913ab38803c57a9,2017-11-03 02:55:58,55.00,34.19
112641,fffb9224b6fc7c43ebb0904318b10b5f,2,43423cdffde7fda63d0414ed38c11a73,b1fc4f64df5a0e8b6913ab38803c57a9,2017-11-03 02:55:58,55.00,34.19
112642,fffb9224b6fc7c43ebb0904318b10b5f,3,43423cdffde7fda63d0414ed38c11a73,b1fc4f64df5a0e8b6913ab38803c57a9,2017-11-03 02:55:58,55.00,34.19


There are orders with an empty "order_delivered_customer_date" column despite being delivered; these are 8 orders, which is only 0.008% of all orders, an extremely small number. This could be due to an ETL/data retrieval error, the timestamp service not running after the order status was updated, human error, or data loss. I checked for these possibilities.

In [ ]:
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


There are orders for which the payment type has not been defined.

In [ ]:
order_payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

These records do not appear to be data quality issues. All undefined payment types belong to canceled orders with a payment value of zero, suggesting that no payment was successfully completed before the cancellation. Therefore, this behavior is consistent with the business process and the records are retained.

In [ ]:
not_defined = pd.merge(
    order_payments[
        order_payments["payment_type"] == "not_defined"
    ],
    orders,
    on="order_id",
    how="left"
)

not_defined[
    [
        "order_id",
        "payment_type",
        "payment_value",
        "order_status"
    ]
]

,order_id,payment_type,payment_value,order_status
0,4637ca194b6387e2d538dc89b124b0ee,not_defined,0.0,canceled
1,00b1cb0320190ca0daa2c88b35206009,not_defined,0.0,canceled
2,c8c528189310eaa44a745b8d9d26908b,not_defined,0.0,canceled
